In [52]:
import pandas as pd
import numpy as np

In [44]:
class Option:
    def __init__(self, strike: float, premium: float, expiry: str):
        self.strike = strike
        self.premium = premium
        self.expiry = expiry
        self.spot_price = None

    def insert_spot(self, spot_price: float):
        self.spot_price = spot_price
    
    def short_call(self):
        # Payoff for short call: premium received - intrinsic value of the option
        intrinsic_value = np.maximum(0, self.spot_price - self.strike)
        return self.premium - intrinsic_value
    
    def long_call(self):
        # Payoff for long call: intrinsic value - premium paid
        intrinsic_value = np.maximum(0, self.spot_price - self.strike)
        return np.maximum(intrinsic_value - self.premium, -self.premium)
    
    def short_put(self):
        # Payoff for short put: premium received - intrinsic value of the option
        intrinsic_value = np.maximum(0, self.strike - self.spot_price)
        return self.premium - intrinsic_value
    
    def long_put(self):
        # Payoff for long put: intrinsic value - premium paid
        intrinsic_value = np.maximum(0, self.strike - self.spot_price)
        return np.maximum(intrinsic_value - self.premium, -self.premium)


class IronCondor(Option):
    def __init__(self, expiry: str):
        self.expiry = expiry
    
    def short_call(self):
        return super().short_call()
    
    def long_call(self):
        return super().long_call()
    
    def short_put(self):
        return super().short_put()
    
    def long_put(self):
        return super().long_put()

    def insert_spot(self, spot_price: float):
        return super().insert_spot(spot_price)

    def long_iron_condor(self, strikes: dict, premiums: dict):
        """
        Long Iron Condor Strategy:
        1. Short 1 OTM Put (strike 1)
        2. Long 1 close to ATM Put (strike 2)
        3. Long 1 close to ATM Call (strike 3)
        4. Short 1 OTM Call (strike 4)
        """
        short_put = Option(strikes['strike1'], premiums['premium1'], self.expiry).short_put()
        long_put = Option(strikes['strike2'], premiums['premium2'], self.expiry).long_put()
        long_call = Option(strikes['strike3'], premiums['premium3'], self.expiry).long_call()
        short_call = Option(strikes['strike4'], premiums['premium4'], self.expiry).short_call()
        payoff = short_put + long_put + long_call + short_call
        return payoff

    def iron_condor(self, direction: str, strikes: dict, premiums: dict):
        """
        Iron Condor Strategy:
        Long (Short) Iron Condor consists of:
        1. Short (Long) 1 OTM Put (strike 1)
        2. Long (Short) 1 close to ATM Put (strike 2)
        3. Long (Short) 1 close to ATM Call (strike 3)
        4. Short (Long) 1 OTM Call (strike 4)
        """
        
        if direction == "long":
            short_put = Option(strikes['strike1'], premiums['premium1'], self.expiry).short_put()
            long_put = Option(strikes['strike2'], premiums['premium2'], self.expiry).long_put()
            long_call = Option(strikes['strike3'], premiums['premium3'], self.expiry).long_call()
            short_call = Option(strikes['strike4'], premiums['premium4'], self.expiry).short_call()
            payoff = short_put + long_put + long_call + short_call
        elif direction == "short":
            long_put = Option(strikes['strike1'], premiums['premium1'], self.expiry).long_put()
            short_put = Option(strikes['strike2'], premiums['premium2'], self.expiry).short_put()
            short_call = Option(strikes['strike3'], premiums['premium3'], self.expiry).short_call()
            long_call = Option(strikes['strike4'], premiums['premium4'], self.expiry).long_call()
            payoff = long_put + short_put + short_call + long_call
        
        return payoff

class OptionsBacktester:
    def __init__(self,
                 signal_df: pd.DataFrame,
                 spot_price_df: pd.DataFrame,
                 options_df: pd.DataFrame,
                 upper_threshold: float,
                 lower_threshold: float,
                 option_type: str,
                 dynamic_option_strategy: bool = False,
                 dynamic_strike: bool = False,):
        self.signal_df = signal_df
        self.signal_df.columns = ["signal"]
        self.spot_price_df = spot_price_df
        self.options_df = options_df
        self.upper_threshold = upper_threshold
        self.lower_threshold = lower_threshold
        self.option_type = option_type
        self.dynamic_option_strategy = dynamic_option_strategy
        self.dynamic_strike = dynamic_strike

        self.comb_df = pd.concat([self.signal_df, self.spot_price_df.shift(), self.options_df.shift()], axis=1)
        self.comb_df["action"] = np.where(self.comb_df["signal"] > self.upper_threshold, 1, np.where(self.comb_df["signal"] < self.lower_threshold, -1, 0))

    def long_short_vol_df(self):
        self.trading_df = {}
        for index, row in self.comb_df.iterrows():
            if row["action"] != 0:
                self.trading_df[index] = self.comb_df.loc[index]

## Code to filter down the options data to only rows where strike matches spot

In [142]:
upper_threshold = 1
lower_threshold = -1

aapl_vrp = pd.read_csv("data/AAPL_vrp_standardised.csv", parse_dates=True, index_col=0)

aapl = pd.read_hdf("data/all_tickers_time_series.hf5", key="AAPL").drop_duplicates().set_index("date")

aapl.index = pd.to_datetime(aapl.index)

options_data = pd.read_hdf("data/all_options_data.h5", key="AAPL").set_index("date")
options_data['exdate'] = pd.to_datetime(options_data['exdate'])
options_data.index = pd.to_datetime(options_data.index)


# options_data_filtered = pd.DataFrame()
options_data['price'] = aapl["prc"]
options_data["strike_price"] = options_data["strike_price"] / 1000
# for date in options_data.index.unique():
#     if options_data.loc[date]["exdate"].nunique() > 1:
#         options_data_filtered = pd.concat([options_data_filtered, options_data.loc[date].query("exdate == exdate.min()")], axis=0)


options_data_filtered = options_data.groupby("date").apply(lambda x: x.query("exdate == exdate.min()")).reset_index(level=0, drop=True)

# options_data_filtered["strike_minus_price"] = np.abs(options_data_filtered["strike_price"] - options_data_filtered["price"])
# new_df = options_data_filtered.groupby("date").apply(lambda x: x.query("strike_minus_price == strike_minus_price.min()")).reset_index(level=0, drop=True)
# date_counts = new_df.index.value_counts()
# def filter_max_open_interest(x):
#     if len(x) >= 2:
#         return x.query("open_interest == open_interest.max()")
#     return x 

# new_new_df = new_df.groupby(['date', 'cp_flag']).apply(filter_max_open_interest).reset_index(level=[0, 1], drop=True)

# comb_df = pd.concat([new_new_df, aapl_vrp], axis=1) # the above block is only used if i want to filter the options data for strike close to spot
comb_df = pd.concat([options_data_filtered, aapl_vrp], axis=1)
comb_df = comb_df.dropna()
comb_df["action"] = np.where(comb_df["vrp_standardised"] > upper_threshold, 1, np.where(comb_df["vrp_standardised"] < lower_threshold, -1, 0))

In [144]:
comb_df

,ticker,exdate,cp_flag,strike_price,best_bid,best_offer,open_interest,impl_volatility,delta,gamma,theta,vega,volume,price,vrp_standardised,action
2020-06-01,AAPL,2020-06-05,C,287.5,34.35,34.75,61.0,0.566173,0.973477,0.003222,-53.77634,2.071664,6.0,321.85001,-0.707107,0
2020-06-01,AAPL,2020-06-05,C,290.0,31.95,32.15,498.0,0.529447,0.971835,0.003623,-52.87457,2.179710,32.0,321.85001,-0.707107,0
2020-06-01,AAPL,2020-06-05,C,292.5,29.45,29.75,129.0,0.513018,0.964684,0.004507,-61.71273,2.622997,6.0,321.85001,-0.707107,0
2020-06-01,AAPL,2020-06-05,C,295.0,27.00,27.20,2438.0,0.475118,0.962196,0.005145,-60.43238,2.773249,1450.0,321.85001,-0.707107,0
2020-06-01,AAPL,2020-06-05,C,297.5,24.50,24.80,183.0,0.453182,0.953750,0.006350,-67.82130,3.268199,17.0,321.85001,-0.707107,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08-31,AAPL,2023-09-01,P,177.5,0.01,0.02,14181.0,0.466922,-0.009593,0.005606,-21.47833,0.253209,3744.0,187.87000,1.165067,1
2023-08-31,AAPL,2023-09-01,P,180.0,0.02,0.03,23962.0,0.392160,-0.017771,0.011368,-30.68410,0.431584,13432.0,187.87000,1.165067,1
2023-08-31,AAPL,2023-09-01,P,182.5,0.04,0.05,20410.0,0.312088,-0.036579,0.026129,-44.56422,0.787846,9587.0,187.87000,1.165067,1
2023-08-31,AAPL,2023-09-01,P,185.0,0.12,0.13,21667.0,0.241647,-0.108708,0.078512,-79.87569,1.832929,34341.0,187.87000,1.165067,1
